# 2장 웹부록 | 우리 반 소셜 그래프 그리기

「영화로 보는 AI 리터러시」 2장 '연결된 세계: 인터넷·빅데이터·4차 산업혁명'의 코랩 실습입니다.

가상의 친구 관계 데이터로 **소셜 그래프**를 그려 봅니다. 가장 연결이 많은 사람, 두 무리를 잇는 '다리' 역할을 하는 사람을 계산으로 찾습니다. 끝에는 '함께 아는 친구' 수로 친구를 추천하는 간단한 추천기를 만듭니다.

**사용법**
- 위에서부터 차례로 각 칸 왼쪽의 ▶(실행) 버튼을 누르세요. 코드를 몰라도 괜찮습니다.
- `# ← 바꿔 보세요`라고 적힌 곳만 고쳐서 다시 ▶를 누르면 됩니다.
- 처음부터 다시 하고 싶으면 메뉴에서 **런타임 → 모두 실행**을 누르세요.

그래프 속 이름은 코랩에서 한글이 깨지지 않도록 영어로 적었습니다. 모두 지어낸 가상의 학생입니다.

## 1단계. 친구 관계 데이터 만들기

책 3-4절의 심화 상식처럼, 사람을 **점**으로, 친구 관계를 **선**으로 나타냅니다.
가상의 한 반 16명입니다. 축구 동아리 무리와 밴드 동아리 무리가 있고, 두 무리를 오가는 학생이 한 명 있습니다.

데이터는 '누구와 누가 친구인가'를 적은 목록일 뿐입니다. 소셜 미디어 회사가 가진 데이터도 크기만 다를 뿐 모양은 같습니다.

👉 **바꿔 볼 곳**: 맨 아래 `NEW_FRIENDS`에 친구 관계를 더해 보세요. 예: `("Minjun", "Yuna")`

In [ ]:
FRIENDS = [
    # 축구 동아리 무리
    ("Minjun", "Seoyeon"), ("Minjun", "Doyun"), ("Minjun", "Haeun"), ("Minjun", "Jiho"),
    ("Minjun", "Siwoo"), ("Seoyeon", "Doyun"), ("Seoyeon", "Haeun"), ("Doyun", "Jiho"),
    ("Haeun", "Siwoo"), ("Jiho", "Siwoo"), ("Doyun", "Jiwoo"), ("Haeun", "Jiwoo"),
    ("Seoyeon", "Jiwoo"),
    # 두 동아리를 오가는 학생
    ("Juwon", "Siwoo"), ("Juwon", "Jiho"), ("Juwon", "Yuna"), ("Juwon", "Hajun"),
    # 밴드 동아리 무리
    ("Yuna", "Hajun"), ("Yuna", "Chaewon"), ("Yuna", "Eunwoo"), ("Hajun", "Chaewon"),
    ("Hajun", "Sua"), ("Chaewon", "Sua"), ("Chaewon", "Yejun"), ("Eunwoo", "Yejun"),
    ("Sua", "Yejun"), ("Sua", "Daeun"), ("Yejun", "Daeun"), ("Eunwoo", "Daeun"),
    ("Seojun", "Eunwoo"), ("Seojun", "Daeun"),
]

NEW_FRIENDS = [
    # ("Minjun", "Yuna"),     # ← 앞의 # 을 지우면 이 친구 관계가 더해집니다
]

edges = FRIENDS + NEW_FRIENDS
friends = {}                       # 사람마다 친구 목록
for a, b in edges:
    friends.setdefault(a, set()).add(b)
    friends.setdefault(b, set()).add(a)
people = sorted(friends)
print(f"사람 {len(people)}명, 친구 관계 {len(edges)}개")
print("Minjun의 친구:", sorted(friends["Minjun"]))

## 2단계. 소셜 그래프 그리기

점의 크기는 친구 수에 비례합니다. 점의 자리는 컴퓨터가 정합니다. 친구끼리는 서로 당기고, 모든 점은 서로 밀어내도록 해서 자리를 잡습니다. 그래서 친한 무리가 저절로 뭉쳐 보입니다.

In [ ]:
import math
import matplotlib.pyplot as plt

def make_layout(people, edges, steps=300, seed=4):
    """친구끼리는 당기고, 모든 점끼리는 밀어내는 '용수철 배치'로 점의 자리를 정합니다."""
    import numpy as np
    rng = np.random.default_rng(seed)
    idx = {p: i for i, p in enumerate(people)}
    xy = rng.normal(size=(len(people), 2))
    k = 1.0
    for step in range(steps):
        diff = xy[:, None, :] - xy[None, :, :]
        d = np.linalg.norm(diff, axis=2) + 1e-9
        move = (diff / d[..., None] * (k * k / d)[..., None]).sum(axis=1)   # 밀어내는 힘
        move -= 2.0 * xy                                                    # 가운데로 모으는 힘
        for a, b in edges:                                                  # 당기는 힘
            i, j = idx[a], idx[b]
            delta = xy[i] - xy[j]
            dist = np.linalg.norm(delta) + 1e-9
            pull = delta / dist * dist * dist / k
            move[i] -= pull
            move[j] += pull
        temp = 0.1 * (1 - step / steps) + 0.005
        length = np.linalg.norm(move, axis=1, keepdims=True) + 1e-9
        xy += move / length * np.minimum(length, temp)
    return {p: tuple(xy[idx[p]]) for p in people}

pos = make_layout(people, edges)

def draw(highlight=None, title="Our class social graph", color_of=None):
    plt.figure(figsize=(10, 5))
    for a, b in edges:
        plt.plot([pos[a][0], pos[b][0]], [pos[a][1], pos[b][1]], color="#bbbbbb", zorder=1)
    for p in people:
        c = (color_of or {}).get(p, "#5b8def")
        if p == highlight:
            c = "#e8743b"
        plt.scatter(*pos[p], s=200 + 120 * len(friends[p]), color=c, edgecolor="white", zorder=2)
        plt.text(pos[p][0], pos[p][1] - 0.22, p, ha="center", fontsize=9)
    plt.title(title)
    plt.axis("off")
    plt.show()

draw()

## 3단계. 연결이 가장 많은 사람 찾기

친구 수를 세면 됩니다. 그래프에서는 이것을 '연결 수'라고 합니다. 네트워크 효과(3-4절)에서 본 것처럼, 연결이 많은 사람은 소식이 가장 빨리 모이는 자리입니다.

In [ ]:
import pandas as pd

degree = {p: len(friends[p]) for p in people}
table = pd.DataFrame({"친구 수": degree}).sort_values("친구 수", ascending=False)
print("친구가 가장 많은 사람:", table.index[0])
table.head(5)

## 4단계. 두 무리를 잇는 '다리' 찾기

친구 수가 많다고 늘 중요한 자리는 아닙니다. 이번에는 **다른 두 사람 사이의 가장 짧은 길에 얼마나 자주 끼어 있는지**를 셉니다.
축구 동아리의 소식이 밴드 동아리로 가려면 반드시 지나가야 하는 사람이 있다면, 그 사람이 '다리'입니다. (전문 용어로 '매개 중심성'이라고 합니다.)

먼저 두 사람 사이의 거리(몇 다리를 건너야 하는지)를 재는 함수를 만들고, 다리 점수를 계산합니다.

In [ ]:
from collections import deque
from itertools import combinations

def shortest_paths(src):
    """src에서 모든 사람까지의 거리와, 가장 짧은 길의 개수를 셉니다 (너비 우선 탐색)."""
    dist, count = {src: 0}, {src: 1}
    queue = deque([src])
    while queue:
        u = queue.popleft()
        for v in friends[u]:
            if v not in dist:
                dist[v], count[v] = dist[u] + 1, 0
                queue.append(v)
            if dist[v] == dist[u] + 1:
                count[v] += count[u]
    return dist, count

info = {p: shortest_paths(p) for p in people}

bridge = {p: 0.0 for p in people}
for s, t in combinations(people, 2):
    ds, cs = info[s]
    dt, ct = info[t]
    if t not in ds:
        continue                                  # 길이 없으면 건너뜁니다
    for v in people:
        if v in (s, t) or v not in ds or v not in dt:
            continue
        if ds[v] + dt[v] == ds[t]:                # v가 s와 t 사이의 가장 짧은 길 위에 있으면
            bridge[v] += cs[v] * ct[v] / cs[t]    # 그 길들 가운데 v를 지나는 몫을 더합니다

table["다리 점수"] = pd.Series(bridge).round(1)
table = table.sort_values("다리 점수", ascending=False)
top_bridge = table.index[0]
print("다리 역할을 가장 많이 하는 사람:", top_bridge)
table.head(6)

In [ ]:
draw(highlight=top_bridge, title=f"Bridge: {top_bridge} (orange)")

친구 수는 많지 않은데 다리 점수는 가장 높은 사람이 있습니다. 이 사람이 빠지면 어떻게 될까요?
아래 칸은 다리 역할을 하는 사람을 빼고, 반이 몇 무리로 나뉘는지 셉니다.

In [ ]:
def groups_without(removed):
    left = [p for p in people if p != removed]
    seen, groups = set(), []
    for p in left:
        if p in seen:
            continue
        group, queue = [], [p]
        seen.add(p)
        while queue:
            u = queue.pop()
            group.append(u)
            for v in friends[u]:
                if v != removed and v not in seen:
                    seen.add(v)
                    queue.append(v)
        groups.append(sorted(group))
    return groups

REMOVE = top_bridge     # ← 다른 이름(예: "Minjun")을 넣어 비교해 보세요
gs = groups_without(REMOVE)
print(f"{REMOVE}이(가) 빠지면 반이 {len(gs)}개 무리로 나뉩니다.")
for g in gs:
    print(" -", g)

## 5단계. 우리 반은 몇 다리면 닿을까

책에서 밀그램의 실험은 '여섯 단계', 2016년 페이스북은 평균 3.57단계였습니다. 우리 반 16명은 평균 몇 단계일까요?

In [ ]:
dists = [info[s][0][t] for s, t in combinations(people, 2) if t in info[s][0]]
print(f"두 사람 사이의 평균 거리: {sum(dists) / len(dists):.2f}단계")
print(f"가장 먼 두 사람 사이: {max(dists)}단계")

## 6단계. '함께 아는 친구'로 친구 추천하기 (그림 2-7)

책의 그림 2-7과 같은 규칙입니다. 나와 아직 친구가 아닌 사람마다, **나와 그 사람이 함께 아는 친구**가 몇 명인지 셉니다. 많은 사람부터 추천합니다.

👉 **바꿔 볼 곳**: `ME`에 다른 이름을 넣어 보세요.

In [ ]:
ME = "Jiwoo"     # ← 추천을 받을 사람 이름을 바꿔 보세요

def recommend(me, top=5):
    scores = []
    for other in people:
        if other == me or other in friends[me]:
            continue                                   # 나 자신과 이미 친구인 사람은 빼고
        mutual = friends[me] & friends[other]          # 함께 아는 친구
        if mutual:
            scores.append((len(mutual), other, sorted(mutual)))
    scores.sort(key=lambda x: (-x[0], x[1]))
    return scores[:top]

print(f"{ME}의 친구: {sorted(friends[ME])}\n")
for rank, (n, other, mutual) in enumerate(recommend(ME), 1):
    print(f"추천 {rank}순위: {other}  (함께 아는 친구 {n}명: {', '.join(mutual)})")

In [ ]:
colors = {p: "#cccccc" for p in people}
for p in friends[ME]:
    colors[p] = "#5b8def"
for n, other, _ in recommend(ME, top=2):
    colors[other] = "#2ca02c"
draw(highlight=ME, color_of=colors, title=f"{ME} (orange), friends (blue), top recommendations (green)")

단순한 규칙이지만, 수십억 명의 그래프에 쓰면 아주 강력해집니다. 같은 그래프가 가짜 뉴스와 감염병이 퍼지는 길을 계산하는 데도 쓰입니다.

## 생각해 볼 것

1. 친구 수가 가장 많은 사람과 다리 점수가 가장 높은 사람이 달랐습니까? 소문이나 가짜 뉴스가 퍼지는 것을 막으려면 어느 쪽이 더 중요할까요?
2. '함께 아는 친구' 추천은 이미 비슷한 사람끼리만 더 묶이게 만들 수 있습니다. 3-7절의 추천 알고리즘, 7절의 '필터 버블'과 연결해 생각해 보세요.
3. 4-2절의 케임브리지 애널리티카 사건에서는 앱에 응답한 사람의 친구들 정보까지 빠져나갔습니다. 이 노트북의 친구 목록만으로 한 사람에 대해 무엇을 짐작할 수 있을까요?